# Tutorial to run benchmark examples, autodiff branch
This notebook demonstrates how to run the benchmark problems in the paper *A Latent-Space
Framework for Stress-Based Multi-Material Topology Optimization* on the `autodiff` branch.
On this branch the gradients of the objective and the constraints are computed by automatic
differentiation (PyTorch autograd) through [PyTO-AD](https://github.com/UW-ERSL/PyTO-AD),
instead of the hand-derived sensitivities on `main`. The examples, settings, schedules and
optimizer are the same.

The entire workflow is described from start to finish: the material database, its principal
components, the latent dimension that follows, the trained decoder, the problem
being solved, how the gradients are computed, a check of those gradients, and the result.

Select the kernel **Python (lsr-stress-mmto-ad)**.

**To run your own problem:** see `CHOICE = 5` in section 1.

| | Domain | Materials | Latent | Objective | Constraints | Elements | Iterations |
|---|---|---|---|---|---|---|---|
| L-bracket | 2D benchmark against Banh et al. | 3 | 2D | p-norm stress | volume fraction per material | 12,825 | 300 |
| U-bracket | 2D, slot corner stress riser | 6 | 2D | p-norm failure factor | mass | 9,001 | 350 |
| Supported beam | 3D octant model | 8 | 2D | p-norm failure factor | mass | 25,748 | 350 |
| Airplane bracket | 3D half model | 20 | 3D | p-norm failure factor | mass and cost | 49,676 | 300 |

Each example is a project file in `projects/`.

## SETUP

In [ ]:
%matplotlib inline
import os

from lsr_mmto.results import records, figures, tables
from lsr_mmto.physics import gradcheck

ROOT = os.path.abspath(".")

import numpy as np
import torch
from lsr_mmto import Project, run
from lsr_mmto.materials.encoder import MaterialEncoder

print(ROOT)

## 1. Choose an example

Set `CHOICE` to one of:

| `CHOICE` | example | problem |
|---|---|---|
| 1 | L-bracket | benchmark against Banh et al., volume fraction per material |
| 2 | U-bracket | six materials, mass constrained |
| 3 | beam | eight materials, octant model |
| 4 | Airplane bracket | twenty alloys, mass and cost constrained |
| 5 | your own problem | a project made in the app, see below |

You may also change `relative_filter_radius` and `material_filter_radius` in section 7 to change the topological and material feature sizes respectively.

### Your own problem

For `CHOICE = 5`, set `MY_PROJECT` in the cell below to the `.mmto` file of your project. Make
the project in the LSR MMTO app (File > New), or with a script exported from it (File > Export
as script). The part, the supports and loads, the database, the objective, the constraints and
the settings all come from that file.

In [ ]:
MY_PROJECT = "./projects/U-bracket.mmto"   # the .mmto file of your own project

In [ ]:
CHOICE = 1                      # 1, 2, 3, 4 or 5

EXAMPLES = {1: "L-bracket", 2: "U-bracket", 3: "beam", 4: "Airplane bracket", 5: "your own problem"}
EXAMPLE = EXAMPLES[CHOICE]

problem = Project.load(MY_PROJECT if CHOICE == 5 else f"./projects/{EXAMPLE}.mmto")
xlsx, _ = problem.database_paths()

print(f"{EXAMPLE}: {problem.comment or problem.stl_path()}")

## 2. The material database

This is the database the VAE is trained on. Each column is a material attribute. 
The VAE embeds these materials in a continuous latent space.

In [ ]:
mesh, mat_prop, bc, body_force, to_params, vae_params = problem.build()

mat_encoder = MaterialEncoder(vae_params)
mat_encoder.readExcel(xlsx)

tables.materials(mat_encoder)

## 3. Latent dimension

The dimension of the latent space is determined using PCA as a heuristic.

PCA on the normalized attributes gives one eigenvalue per direction. The
effective dimension $d_{\mathrm{eff}}$ is the participation ratio of those
eigenvalues, a non-integer count of how many directions carry real variance.
The latent dimension is $n_z = \lceil d_{\mathrm{eff}} \rceil$.

In [ ]:
tables.pca(mat_encoder)

## 4. The problem

Red arrows are the applied load, black markers the fixed and symmetry degrees
of freedom, drawn on the meshed design domain.

In [ ]:
import pyvista as pv
pv.set_jupyter_backend("static")
pv.global_theme.window_size = [1400, 1050]
pv.global_theme.anti_aliasing = "ssaa"
_ = figures.boundary_conditions(problem, size=(920, 675), zoom=0.9)


## 5. The decoder

Set `USE_REFERENCE_VAE = True` to load the VAE used for this benchmark in the
paper. Set it to `False` to train one from scratch.

A freshly trained VAE gives a different latent space, and so a different
design. The run loads whichever network this cell prepares, so the
reconstruction error below is for the decoder it uses.

In [ ]:
import os

USE_REFERENCE_VAE = True

if USE_REFERENCE_VAE:
    net = xlsx.replace(".xlsx", ".nt")        # the network shipped with the database
    mat_encoder.loadAutoencoderFromFile(net)
    print(f"loaded {net}")
else:
    os.makedirs("results", exist_ok=True)
    net = f"results/{EXAMPLE}_trained.nt"
    torch.manual_seed(1234)
    mat_encoder.trainAutoencoder(
        vae_params.numEpochs, vae_params.klFactor, net,
        vae_params.learningRate, vae_params.maxAttributeErrorPercent)
    print(f"trained a fresh network, saved to {net}")

### How faithfully the decoder reproduces the database

Every real material is encoded to its latent point and decoded back. The error reported in the table below
is the difference between the decoded normalized attribute and the real normalized attribute from the database for that material. It measures how much the
embedding distorts the catalogue.

In [ ]:
tables.encoding_errors(mat_encoder)

## 6. How the gradients are computed

The objective, the constraints and the two penalties are computed as one chain of tensor
operations from the design variables, the density $x$ and the latents $z$:

| quantity | depends on |
|---|---|
| $x_{phys}$ | $x$, through the density filter and the Heaviside projection |
| $E$, $\nu$, and $Y$, $\rho$ where the database has them | $z$, through the decoder |
| displacements $\mathbf{u}$ | $x_{phys}$, $E$, $\nu$, through $\mathbf{K}\mathbf{u} = \mathbf{f}$ |
| element stress, von Mises stress $\sigma_{vm}$ | $\mathbf{u}$, $x_{phys}$, $E$, $\nu$ |
| objective $J$ | $\sigma_{vm}$, and $Y$ for the failure factor |
| constraints | $x_{phys}$ and $z$ (volume fraction per material), $\rho$ (mass) |
| penalties | $x_{phys}$, $z$ |

Autograd follows this chain back from each quantity to $x$ and $z$. The one step it cannot
follow by itself is the linear solve. PyTO-AD's sparse solve supplies that step: one adjoint
solve $\mathbf{K}^T \boldsymbol{\lambda} = \partial J / \partial \mathbf{u}$, reusing the
factorization of the forward solve.

The hand-derived sensitivities of `main` (`SensitivitiesPureStructural.py`) are not used on this
branch. The model is `MMTOModelAD` in `lsr_mmto/physics/structural.py`.

## 7. Run settings

The settings are stored in the project file; for the four examples they are the ones reported
in the paper. Every setting and what it does is in `docs/project_reference.md`. The gradient
check in section 8 and the run in section 9 both use them.

In [ ]:
problem.settings["vae"].update(saveNet=os.path.abspath(net), use_pretrained_vae=True)   # the network of section 5

# any setting can be changed here, for example:
# problem.settings["common"]["relative_filter_radius"] = 3.5

## 8. Check the gradient

At a random design, the gradient of the objective $J$ and of the first constraint $g_1$ is
compared with central finite differences on a few density and latent variables, for three
step sizes $h$. It takes a minute or two.

The difference has two sources: truncation error, which shrinks with $h$, and round-off in
the solve, which grows as $h$ shrinks. The best agreement is therefore at an intermediate $h$.
A gap that stays the same for every $h$ would mean a wrong gradient.

In [ ]:
gradcheck.check(problem)

## 9. Run

`run` saves the result in `results/<example>`: the record (`run_data.npz`), and the run's settings with the project as it was run (`run_data.config.json`, and `run_config.yaml` to read by eye).

In [ ]:
run(problem, save_dir=f"results/{EXAMPLE}")

result = records.load(f"results/{EXAMPLE}/run_data.npz")
print(result.descriptor)

## 10. The result

In [ ]:
tables.summary(result)

### Optimized design

Coloured by material after binarization.

In [ ]:
_=figures.design(result)

### Latent field

Black stars are the real materials, red points the converged latent coordinates
of the solid elements. Points sitting on a star are elements that resolved onto
a real material; points between them have not.

In [ ]:
_=figures.latent_scatter(result)

### Convergence

The objective on the left axis, each constraint on the right as a fraction of
its budget.

In [ ]:
_=figures.convergence(result)